# dialog

> Pure functions that turn a Jupyter notebook (as JSON) into a chat dialog — no JupyterLab, no OpenAI required.

Since v0.2 yasi sends the *whole* notebook to the model by default (solveit-style): untagged markdown cells and code cells (including their outputs) become `user` messages, while the classic role tags (`#| chat_system`, `#| chat_user`, `#| chat_assistant`) still override the role of any cell. The old tagged-only behavior is available with `all_cells=False`.

In [ ]:
#| default_exp dialog

In [ ]:
#| hide
from nbdev.showdoc import *

## Tag detection

In [ ]:
#| export
def tag_in_cell(cell, # Dictonary of a Jupyter Notebook cell
                tag # The tag to search
               )->bool: # True if any line contains the given tag
    """Checks a Jupyter Notebook cells source, if any line starts with the given tag"""
    return any([line.startswith(tag) for line in cell['source']])

In [ ]:
assert tag_in_cell({'source': ['#| chat_user\n', 'Hi']}, '#| chat_user')
assert not tag_in_cell({'source': ['plain text\n']}, '#| chat_user')

In [ ]:
#| export
def cell_roles(cell, # Dictonary of a Jupyter Notebook cell
               tags:dict # Mapping of role name to tag, e.g. {'user': '#| chat_user'}
              )->list: # Roles whose tag was found in the cell
    """Returns all dialog roles whose tag appears in the given cell"""
    return [role for role, tag in tags.items() if tag_in_cell(cell, tag)]

## Rendering code cell outputs

Code cell outputs come in four flavors (`stream`, `execute_result`, `display_data`, `error`). We render them as plain text: tracebacks get their ANSI color codes stripped, non-text data (e.g. base64 images) is replaced by a short note, and overly long output is truncated so a single cell cannot flood the context.

In [ ]:
#| export
import re

_ANSI_RE = re.compile(r'\x1b\[[0-9;]*[a-zA-Z]')

def strip_ansi(text:str # Text possibly containing ANSI escape sequences
              )->str: # Text without ANSI escape sequences
    """Removes ANSI escape sequences (e.g. colors in tracebacks) from text"""
    return _ANSI_RE.sub('', text)

In [ ]:
assert strip_ansi('\x1b[0;31mZeroDivisionError\x1b[0m') == 'ZeroDivisionError' 

In [ ]:
#| export
def render_output(output:dict, # A single entry of a code cell's 'outputs' list
                  skip_note:str='[non-text output skipped]' # Placeholder for non-text data
                 )->str: # Plain text representation
    """Renders one code cell output as plain text"""
    output_type = output.get('output_type')
    if output_type == 'stream':
        return ''.join(output.get('text', []))
    if output_type in ('execute_result', 'display_data'):
        data = output.get('data', {})
        if 'text/plain' in data: return ''.join(data['text/plain'])
        return skip_note
    if output_type == 'error':
        return strip_ansi('\n'.join(output.get('traceback', [])))
    return ''

def render_outputs(outputs:list, # A code cell's 'outputs' list
                   max_len:int=2000 # Maximum number of characters, longer output is truncated
                  )->str: # Plain text representation of all outputs
    """Renders all outputs of a code cell as plain text, truncated to `max_len` characters"""
    rendered = '\n'.join([r for r in (render_output(o) for o in outputs) if r.strip()])
    if max_len is not None and len(rendered) > max_len:
        rendered = rendered[:max_len] + '\n... [output truncated]'
    return rendered.strip()

In [ ]:
outputs = [
    {'output_type': 'stream', 'name': 'stdout', 'text': ['hello\n']},
    {'output_type': 'execute_result', 'data': {'text/plain': ['42']}},
    {'output_type': 'display_data', 'data': {'image/png': 'iVBORw0...'}},
    {'output_type': 'error', 'ename': 'ZeroDivisionError', 'evalue': 'division by zero',
     'traceback': ['\x1b[0;31mZeroDivisionError\x1b[0m: division by zero']},
]
rendered = render_outputs(outputs)
assert 'hello' in rendered and '42' in rendered
assert '[non-text output skipped]' in rendered
assert 'ZeroDivisionError: division by zero' in rendered and '\x1b' not in rendered
assert render_outputs([{'output_type': 'stream', 'text': ['x'*100]}], max_len=10).endswith('[output truncated]')
print(rendered)

In [ ]:
#| export
def code_cell_content(cell:dict, # Dictonary of a Jupyter Notebook code cell
                      max_output_len:int=2000 # Maximum characters of rendered output
                     )->str: # Markdown-formatted message content
    """Turns a code cell into message content: fenced source plus rendered outputs"""
    content = f"```python\n{''.join(cell['source'])}\n```"
    rendered = render_outputs(cell.get('outputs', []), max_len=max_output_len)
    if rendered:
        content += f"\n\nOutput:\n```\n{rendered}\n```"
    return content

## Dialog extraction

`extract_dialog` walks all cells of a notebook and builds the `messages` list for the chat completions API.

- Cells with exactly one role tag are sent with that role (any cell type), exactly as before.
- With `all_cells=True` (the default) untagged markdown cells are sent as `user` messages and untagged code cells are sent as `user` messages containing their source and outputs. Raw and empty cells are skipped.
- With `all_cells=False` untagged cells are ignored (the pre-v0.2 behavior).
- Cells carrying more than one tag are ambiguous; for those a warning text is returned so the caller can surface it (yasi inserts it as a new markdown cell).

## Prompt cells

A *prompt cell* is a code cell starting with the `%%prompt` magic (see `yasi.magics`). During extraction they are treated as plain user messages: the magic line is stripped and their outputs are ignored (the response lives in the markdown cell yasi inserts below).

In [ ]:
#| export
PROMPT_MAGIC = '%%prompt'

def is_prompt_cell(cell:dict # Dictonary of a Jupyter Notebook cell
                  )->bool: # True if the cell is a code cell starting with the %%prompt magic
    """Checks whether a cell is a yasi prompt cell"""
    if cell['cell_type'] != 'code': return False
    return ''.join(cell['source']).lstrip().startswith(PROMPT_MAGIC)

def prompt_body(cell:dict # Dictonary of a Jupyter Notebook prompt cell
               )->str: # The prompt text without the magic line
    """Returns the source of a prompt cell without the leading %%prompt line"""
    lines = ''.join(cell['source']).lstrip().split('\n')
    return '\n'.join(lines[1:]).strip()

def notebook_upto_prompt(notebook:dict, # Jupyter Notebook JSON as a dictionary
                         body:str # Body of the currently executing prompt cell
                        )->tuple: # (truncated notebook dict, found flag)
    """Truncates the notebook after the first prompt cell whose body matches `body`"""
    cells, found = [], False
    for cell in notebook['cells']:
        cells.append(cell)
        if is_prompt_cell(cell) and prompt_body(cell) == body.strip():
            found = True
            break
    return {'cells': cells}, found

In [ ]:
pcell = {'cell_type': 'code', 'source': ['%%prompt\n', 'What is money?'], 'outputs': [{'output_type': 'stream', 'text': ['noise']}]}
assert is_prompt_cell(pcell)
assert not is_prompt_cell({'cell_type': 'markdown', 'source': ['%%prompt']})
assert prompt_body(pcell) == 'What is money?'

nb_fixture = {'cells': [
    {'cell_type': 'markdown', 'source': ['notes']},
    pcell,
    {'cell_type': 'markdown', 'source': ['below, must be cut off']},
]}
truncated, found = notebook_upto_prompt(nb_fixture, 'What is money?')
assert found and len(truncated['cells']) == 2
_, found2 = notebook_upto_prompt(nb_fixture, 'no such prompt')
assert not found2

In [ ]:
#| export
def multiple_tags_warning(cell # Dictonary of a Jupyter Notebook cell
                         )->str: # Markdown warning text
    """Builds the warning text for a cell that contains more than one dialog tag"""
    tmp_content = '## \u26a0\u26a0\u26a0 cell contains multiple tags \u26a0\u26a0\u26a0\n'
    tmp_content += 'identify the following cell and select only one tag\n'
    tmp_content += '> ' + '> '.join(cell['source'])
    return tmp_content

In [ ]:
#| export
def extract_dialog(notebook:dict, # Jupyter Notebook JSON as a dictionary
                   tag_system:str='#| chat_system', # tag for system chat markdown cells
                   tag_user:str='#| chat_user', # tag for user chat markdown cells
                   tag_assistant:str='#| chat_assistant', # tag for assistant chat markdown cells
                   all_cells:bool=True, # send untagged markdown/code cells as user messages
                   max_output_len:int=2000 # maximum characters of rendered output per code cell
                  )->tuple: # (messages, warnings)
    """Extracts the dialog from a notebook dictionary and turns it into a messages list.
    Returns the messages plus warning texts for cells containing multiple tags."""
    tags = {'system': tag_system, 'user': tag_user, 'assistant': tag_assistant}
    messages, warnings = [], []
    for cell in notebook['cells']:
        roles = cell_roles(cell, tags)
        if len(roles) > 1:
            warnings.append(multiple_tags_warning(cell))
            continue
        source = ''.join(cell['source'])
        if len(roles) == 1:
            messages.append({"role": roles[0], "content": source})
            continue
        if not all_cells or not source.strip():
            continue
        if is_prompt_cell(cell):
            messages.append({"role": "user", "content": prompt_body(cell)})
        elif cell['cell_type'] == 'code':
            messages.append({"role": "user", "content": code_cell_content(cell, max_output_len=max_output_len)})
        elif cell['cell_type'] == 'markdown':
            messages.append({"role": "user", "content": source})
    return messages, warnings

In [ ]:
sample_nb = {'cells': [
    {'cell_type': 'markdown', 'source': ['#| chat_system\n', '\n', 'You are a helpful assistant.']},
    {'cell_type': 'markdown', 'source': ['# My analysis notes']},
    {'cell_type': 'code', 'source': ['x = 6*7\n', 'x'],
     'outputs': [{'output_type': 'execute_result', 'data': {'text/plain': ['42']}}]},
    {'cell_type': 'code', 'source': ['   '], 'outputs': []},
    {'cell_type': 'raw', 'source': ['raw cells are skipped']},
    {'cell_type': 'markdown', 'source': ['#| chat_user\n', '\n', 'Why 42?']},
    {'cell_type': 'markdown', 'source': ['#| chat_assistant\n', '\n', 'Because Adams.']},
    {'cell_type': 'markdown', 'source': ['#| chat_user\n', '#| chat_assistant\n', 'oops, two tags']},
]}

# solveit-style default: everything visible goes to the model
messages, warnings = extract_dialog(sample_nb)
assert [m['role'] for m in messages] == ['system', 'user', 'user', 'user', 'assistant']
assert messages[1]['content'] == '# My analysis notes'
assert messages[2]['content'].startswith('```python\nx = 6*7')
assert 'Output:\n```\n42\n```' in messages[2]['content']
assert len(warnings) == 1 and 'multiple tags' in warnings[0]

# classic tagged-only behavior
messages_old, _ = extract_dialog(sample_nb, all_cells=False)
assert [m['role'] for m in messages_old] == ['system', 'user', 'assistant']

# prompt cells are sent as plain user messages, outputs ignored
nb_p = {'cells': [{'cell_type': 'code', 'source': ['%%prompt\n', 'Why 42?'],
                   'outputs': [{'output_type': 'stream', 'text': ['ignored']}]}]}
msgs_p, _ = extract_dialog(nb_p)
assert msgs_p == [{'role': 'user', 'content': 'Why 42?'}]
messages

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()